# 🔌 Model Context Protocol (MCP) — Build Your First MCP Server

### Dinesh AI Academy | Day 4 — Agents & MCP

**Learning objective:**
By the end of this notebook you can explain, in plain English, what an MCP server is, build a working one in Python, connect a real client to it, plug it into an LLM tool-calling loop, and judge when MCP is (and isn't) the right tool for the job.

### Roadmap for this notebook

```text
1. Recap: plain tool calling (Day 3)
2. The problem MCP solves
3. What an MCP server actually is
4. MCP architecture: Host, Client, Server
5. Build a real MCP server in Python
6. Connect a real client to it
7. Tools vs Resources
8. Bridge back to Day 3: let an LLM drive your MCP server
9. How real apps (Claude Desktop / Claude Code / Cursor) integrate MCP servers
10. When to use MCP — and when NOT to
```


## 0. Quick recap — plain tool calling (Day 3)

Yesterday we saw this loop:

```text
User message
     ↓
LLM  (sees the user message + tool definitions)
     ↓
LLM generates a structured function call, e.g. calculate(a=25, b=40, operation="multiply")
     ↓
OUR PYTHON APPLICATION executes the function directly
     ↓
Result goes back to the LLM
     ↓
Final answer
```

The tool's Python function lived **inside the same app** as the LLM call. That's perfectly fine for a single app with a handful of local functions.

The question Day 4 answers is: **what happens when you want the same tools reusable across many different AI apps** (a chat app, a CLI, an IDE assistant, a Slack bot...) — without rewriting `calculate()` and `get_weather()` inside every single one of them?

That's the problem MCP was built to solve.


## 1. The problem MCP solves

Imagine you have:

- **3 AI applications**: a chatbot, a coding assistant, a CLI agent
- **4 capabilities** you want them all to use: a file system, a database, GitHub, Slack

**Without a standard**, every app needs its own custom integration code for every capability:

```text
Chatbot     ──custom code──▶ Filesystem
Chatbot     ──custom code──▶ Database
Chatbot     ──custom code──▶ GitHub
Chatbot     ──custom code──▶ Slack
Coding tool ──custom code──▶ Filesystem
Coding tool ──custom code──▶ Database
... and so on
```

That's **3 apps × 4 capabilities = 12 separate integrations** to write and maintain. This is often called the **M×N integration problem**.

**With MCP**, each capability is wrapped once as an MCP *server*, and each app only needs to speak the *one* MCP protocol as a *client*:

```text
Chatbot     ─┐
Coding tool  ─┼── MCP (one protocol) ──▶  Filesystem server
CLI agent   ─┘                       ──▶  Database server
                                      ──▶  GitHub server
                                      ──▶  Slack server
```

That's **3 + 4 = 7** things to build instead of 12, and it keeps growing better as you add more apps or more capabilities.

| | Without MCP | With MCP |
|---|---|---|
| Integration effort | M × N custom integrations | M clients + N servers |
| Reusing a tool in a new app | Rewrite the integration | Just connect the existing server |
| Who maintains the tool logic | Duplicated in every app | Centralized in one server |

People often describe MCP as **"a USB-C port for AI applications"** — one standard connector that any compliant app (the "host") can plug into any compliant tool provider (the "server"), instead of a different custom cable for every combination.


## 2. What exactly is an MCP server?

> **MCP (Model Context Protocol)** is an open, standardized protocol that lets an application expose data and capabilities to an LLM app in a consistent way — think of it like a small, purpose-built web API, but designed specifically for LLM tool use.

An **MCP server** is just a program that:

1. Exposes one or more capabilities using the MCP protocol
2. Waits for a **client** to connect to it
3. Responds to requests like "what tools do you have?" or "run this tool with these arguments"

An MCP server can expose three kinds of things:

| Primitive | What it is | Analogy | Example |
|---|---|---|---|
| **Tool** | A function the LLM can *call* to take an action or compute something | A `POST` endpoint | `add_note(title, content)` |
| **Resource** | Read-only data the app can *fetch* and give to the LLM as context | A `GET` endpoint | `note://project-plan` |
| **Prompt** | A reusable, parameterized prompt template the host can offer to the user | A saved template | "Summarize this note in a {style} tone" |

For this notebook, we'll focus mostly on **Tools** (the direct continuation of Day 3), and touch **Resources** briefly.


## 3. MCP architecture — Host, Client, Server

```text
┌────────────────────────────────────────────────────┐
│  HOST APPLICATION  (e.g. Claude Desktop, an IDE,     │
│                      your own chatbot)               │
│                                                        │
│   ┌───────────────┐        ┌───────────────┐         │
│   │  MCP CLIENT A │        │  MCP CLIENT B │         │
│   └───────┬───────┘        └───────┬───────┘         │
└───────────┼────────────────────────┼─────────────────┘
            │ MCP protocol           │ MCP protocol
            ▼                        ▼
   ┌─────────────────┐      ┌─────────────────┐
   │  MCP SERVER 1    │      │  MCP SERVER 2   │
   │  (e.g. Notes)    │      │  (e.g. GitHub)  │
   └─────────────────┘      └─────────────────┘
```

- **Host** — the AI application the human actually uses (Claude Desktop, Claude Code, your own app). It decides what the LLM sees and manages one or more clients.
- **Client** — lives inside the host, and maintains **one dedicated connection to exactly one server**. It speaks the MCP protocol on the host's behalf.
- **Server** — a separate, standalone program that exposes tools/resources/prompts. It has no idea which LLM (if any) is driving the conversation — it just answers protocol requests.

A **client and server exchange messages over a *transport***. The two you'll meet most often:

| Transport | Typical use |
|---|---|
| **stdio** | Server runs as a local subprocess; client talks to it over stdin/stdout. Used by Claude Desktop, Claude Code, Cursor for local servers. |
| **Streamable HTTP** | Server runs independently (e.g. on a remote machine) and exposes an HTTP endpoint. Used for shared/remote servers. |

The important idea: **the server code never changes between transports or between hosts.** The same `mcp_server.py` we're about to write can be used by Claude Desktop, Claude Code, or our own Python script below.


## 4. Install the MCP Python SDK

We'll use the official Python SDK, published on PyPI as `mcp`. The `[cli]` extra also gives us the `mcp` command-line tool (handy for quick manual testing).

In [1]:
!pip -q install "mcp[cli]" 

## 5. Build your first MCP server

A few things to notice in the code below:

- `MCPServer("NotesServer")` creates the server and gives it a name.
- `@mcp.tool()` turns a normal Python function into an MCP tool. The **type hints become the tool's input schema**, and the **docstring becomes the tool's description** — exactly like the tool declarations we hand-wrote for Gemini on Day 3. A vague docstring makes it harder for an LLM to pick the right tool later, so write it like you're explaining the function to someone who can only read one sentence.
- `@mcp.resource("note://{title}")` exposes readable data at a URI, instead of an action to run.
- `mcp.run()` starts the server. With no arguments, it listens on **stdio** — the same way Claude Desktop, Claude Code, and Cursor talk to local servers.

We write this to a real file because an MCP server is meant to run as its **own separate process** — that separation is the whole point of the architecture.

In [2]:
%%writefile mcp_server.py
"""A tiny MCP server that manages study notes for the bootcamp."""

from mcp.server import MCPServer

mcp = MCPServer("NotesServer")

# In-memory storage — resets every time the server process restarts.
_notes: dict[str, str] = {}


@mcp.tool()
def add_note(title: str, content: str) -> str:
    """Save a note under a title. Overwrites any existing note with the same title."""
    _notes[title] = content
    return f"Saved note '{title}' ({len(content)} characters)."


@mcp.tool()
def search_notes(keyword: str) -> list[str]:
    """Search saved notes for a keyword (case-insensitive) and return the matching titles."""
    keyword = keyword.lower()
    return [
        title for title, content in _notes.items()
        if keyword in title.lower() or keyword in content.lower()
    ]


@mcp.resource("note://{title}")
def get_note(title: str) -> str:
    """Fetch the full content of a single saved note by its title."""
    return _notes.get(title, f"No note found with title '{title}'.")


if __name__ == "__main__":
    # Default transport is stdio: read requests from stdin, write responses to stdout.
    # This is exactly how Claude Desktop / Claude Code launch local MCP servers.
    mcp.run()


Overwriting mcp_server.py


## 6. (Optional, local only) Try it visually with the MCP Inspector

Before writing any client code, the MCP CLI gives you a zero-code way to poke at a server: the **MCP Inspector**, a web UI that lists your tools/resources and lets you call them by hand.

Run this in a terminal **on your own machine** (it opens a browser tab, so it won't work inside a hosted Colab runtime):

```bash
mcp dev mcp_server.py
```

This is the fastest way to sanity-check a server while you're building it — always try this before wiring up a full client or LLM integration.

## 7. Connect a real client to your server

Now let's write an actual MCP **client** and talk to the server we just wrote — the same way a host application like Claude Desktop would.

- `StdioServerParameters` tells the client **how to start the server**: which command to run and with what arguments. The client spawns `mcp_server.py` as a subprocess and talks to it over stdin/stdout.
- `Client(...)` opens the connection. It also accepts a plain URL string instead, if you're connecting to a server running over HTTP.
- `list_tools()` asks the server what it can do — this is the same information an LLM host would show the model.
- `call_tool(name, arguments)` actually invokes a tool and gets the result back.

Notice: **nothing here mentions an LLM.** The client/server conversation is just structured protocol messages — the LLM only enters the picture later, in section 8, when something decides *which* tool to call.

In [3]:
import sys
from mcp import Client, StdioServerParameters

server_params = StdioServerParameters(
    command=sys.executable,   # the same Python interpreter running this notebook
    args=["mcp_server.py"],
)

async def explore_server():
    async with Client(server_params) as client:
        tools = await client.list_tools()
        print("Tools exposed by the server:")
        for t in tools.tools:
            print(f"  • {t.name:15} — {t.description}")

        print()
        result = await client.call_tool(
            "add_note",
            {"title": "day4", "content": "MCP servers expose tools over a standard protocol."},
        )
        print("add_note result:", result.structured_content)

        result = await client.call_tool("search_notes", {"keyword": "protocol"})
        print("search_notes result:", result.structured_content)

        note = await client.read_resource("note://day4")
        print("note://day4 resource:", note.contents[0].text)

await explore_server()


UnsupportedOperation: fileno

## 8. Tools vs Resources — what just happened?

Look back at the three calls we made:

| Call | Primitive | What it did |
|---|---|---|
| `call_tool("add_note", ...)` | **Tool** | Took an *action* — it changed server state (saved a note) |
| `call_tool("search_notes", ...)` | **Tool** | *Computed* something and returned a result |
| `read_resource("note://day4")` | **Resource** | *Fetched* existing data by its URI, with no side effects |

A simple rule of thumb: if the LLM is asking "**do this**", it's a tool call. If the LLM (or the host app) is asking "**give me this data as context**", it's a resource read.


## 9. Bridge to Day 3 — let an LLM drive your MCP server

This is the real payoff. Day 3's loop was:

```text
LLM picks a tool  →  OUR APPLICATION runs a local Python function
```

Today's loop is nearly identical — we just swap out *who executes the tool*:

```text
LLM picks a tool  →  OUR APPLICATION calls it on the MCP SERVER instead of locally
```

The Gemini side barely changes from Day 3. We describe the same two tools as `FunctionDeclaration`s (in a production system you'd generate these automatically from `list_tools()`'s schemas — we're mapping them by hand so the connection stays visible), let Gemini pick one, and then execute it through our **MCP client** instead of a local Python function.

In [ ]:
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

print("✅ API key loaded successfully.")


In [ ]:
!pip -q install -U google-genai

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GAISTUDIO_API_KEY)
MODEL = "gemini-3.6-flash"

add_note_declaration = types.FunctionDeclaration(
    name="add_note",
    description="Save a note under a title. Overwrites any existing note with the same title.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "title": types.Schema(type=types.Type.STRING, description="Note title"),
            "content": types.Schema(type=types.Type.STRING, description="Note content"),
        },
        required=["title", "content"],
    ),
)

search_notes_declaration = types.FunctionDeclaration(
    name="search_notes",
    description="Search saved notes for a keyword (case-insensitive) and return the matching titles.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "keyword": types.Schema(type=types.Type.STRING, description="Keyword to search for"),
        },
        required=["keyword"],
    ),
)

gemini_tools = types.Tool(function_declarations=[add_note_declaration, search_notes_declaration])
config = types.GenerateContentConfig(tools=[gemini_tools], temperature=0)


async def ask_agent(user_prompt: str):
    async with Client(server_params) as mcp_client:
        response = client.models.generate_content(model=MODEL, contents=user_prompt, config=config)

        function_call = None
        for part in response.candidates[0].content.parts:
            if part.function_call:
                function_call = part.function_call
                break

        if not function_call:
            print("Gemini answered directly:", response.text)
            return

        print("Gemini selected MCP tool:", function_call.name, dict(function_call.args))

        # Execute the call on the MCP SERVER instead of a local Python function.
        tool_result = await mcp_client.call_tool(function_call.name, dict(function_call.args))

        follow_up = client.models.generate_content(
            model=MODEL,
            contents=[
                types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
                response.candidates[0].content,
                types.Content(
                    role="tool",
                    parts=[types.Part.from_function_response(
                        name=function_call.name,
                        response={"result": tool_result.structured_content},
                    )],
                ),
            ],
            config=config,
        )
        print("Final answer:", follow_up.text)


await ask_agent("Save a note titled 'mcp' with the content: MCP standardizes how AI apps use external tools.")
await ask_agent("Search my notes for anything about 'standardizes'.")


## 10. How real apps integrate an MCP server

The whole point of building `mcp_server.py` the way we did is that **any** MCP-compatible host can now use it, unmodified — not just our notebook.

**Claude Desktop / Claude Code / Cursor** all use roughly the same idea: a config file that tells the host which command starts your server (they connect over stdio, exactly like our client did):

```json
{
  "mcpServers": {
    "notes": {
      "command": "python",
      "args": ["/absolute/path/to/mcp_server.py"]
    }
  }
}
```

Claude Code also lets you register one from the terminal:

```bash
claude mcp add notes -- python /absolute/path/to/mcp_server.py
```

Once registered, the host starts your server as a subprocess, calls `list_tools()` behind the scenes, and offers those tools to the model — the exact same two calls we made by hand in section 7. **Nothing in `mcp_server.py` needed to change.** That reusability is the entire value proposition of MCP versus Day 3's local-function tool calling.


## 11. When SHOULD you build/use an MCP server?

- You want the **same tool reused across multiple apps** (your chatbot, your IDE assistant, a teammate's CLI agent) without copy-pasting integration code.
- You're exposing a **real backend, database, or external service** and want one governed, testable place that owns that access — instead of scattering API keys and query logic across every app that needs it.
- You want a tool that's **swappable**: point any MCP host at a different server without touching the host's code.
- You need more than function calls — you also want to offer **resources** (read-only context) or **prompts** (reusable templates) to the host.
- You're building something meant to plug into **existing MCP hosts** (Claude Desktop, Claude Code, Cursor, etc.) rather than only your own app.

## 12. When should you NOT bother with MCP?

- **One app, one-off function, never reused elsewhere.** Day 3's plain tool calling is simpler — no separate process, no protocol overhead, no extra file to maintain.
- **You need the lowest possible latency** for an in-process call. Spawning/talking to a subprocess (or an HTTP hop) is strictly slower than calling a Python function directly.
- **You're prototyping fast** and don't yet know if this capability needs to be shared across apps. Start local; promote to an MCP server once reuse becomes real, not hypothetical.
- **The "tool" is trivial** (e.g. string formatting, a basic calculation) — wrapping it in a server adds ceremony without adding value.

> **Rule of thumb:** reach for MCP when the *tool* needs to outlive and outreach a single app. Reach for plain tool calling when the tool and the app are always going to be the same thing.


## 13. Challenge

Extend `mcp_server.py` with one more capability, then re-run section 7 or 9 to see it picked up automatically (no client code changes needed):

1. Add a `delete_note(title: str) -> str` tool.
2. Add a `@mcp.prompt()` named `summarize_note` that takes a `title` and returns a prompt string asking an LLM to summarize that note.
3. Ask a student: *what changed in the client code to support this new tool?* (Answer: nothing — `list_tools()` picks it up automatically. That's the point.)


# 🎯 Day 4 Takeaway

Students should leave this notebook understanding these six ideas:

1. **MCP is a standard protocol**, not a specific tool — it solves the M×N integration problem between AI apps and capabilities.
2. **A host contains clients; each client talks to exactly one server** over a transport (stdio for local, Streamable HTTP for remote).
3. **An MCP server exposes tools, resources, and prompts** — `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()` — with docstrings and type hints doing the same descriptive job they did for Gemini function declarations on Day 3.
4. **The LLM still only *decides* which tool to call** — the MCP server, like our local Python functions on Day 3, is what actually executes it.
5. **The same server works unmodified across many hosts** — our notebook, Claude Desktop, Claude Code, and Cursor can all launch and use `mcp_server.py` as-is.
6. **MCP isn't always the right choice** — it earns its overhead when a tool needs to be shared and reused; a single app with a local function is often better served by plain Day-3-style tool calling.

### One sentence to remember

> **MCP standardizes how AI apps discover and call tools, so a capability you build once can be reused by any MCP-compatible app — but that standardization is only worth its overhead when reuse is real.**

### Next step

**Tools → Workflows → Agents → MCP → Multi-agent systems (Day 5)**


## Official references

- Model Context Protocol docs: https://modelcontextprotocol.io
- MCP Python SDK (GitHub): https://github.com/modelcontextprotocol/python-sdk
- MCP Inspector: https://github.com/modelcontextprotocol/inspector
- Gemini Function Calling: https://ai.google.dev/gemini-api/docs/function-calling
- Google AI Studio: https://aistudio.google.com/
